# Brisbane Urban Heat & Canopy Gap Analysis
## Identifying Priority Suburbs for Street Tree Planting

This notebook analyses the relationship between urban heat intensity and tree canopy 
coverage across Brisbane suburbs to identify where new street trees would have the 
greatest impact.

**Data Sources:**
- Land Surface Temperature: Landsat 8 Collection 2, USGS via Google Earth Engine (summers 2022–2024)
- Tree Canopy: BCC Protected Vegetation (Natural Assets Local Law 2003)
- Suburb Boundaries: ABS Statistical Area Level 1, GDA2020

**Methodology:**
1. Extract mean summer LST per suburb via zonal statistics
2. Calculate protected canopy coverage % per suburb
3. Normalise both metrics and compute a composite priority score
4. Identify suburbs with high heat AND low canopy as priority planting zones

In [1]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
from folium.features import GeoJsonTooltip
from rasterstats import zonal_stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load Data
Loading preprocessed suburb boundaries, vegetation polygons, and Landsat LST raster.

In [2]:
suburbs = gpd.read_file("../data/brisbane_suburbs.gpkg")
vegetation = gpd.read_file("../data/vegetation_brisbane.gpkg")

print(f"Suburbs loaded: {len(suburbs)}")
print(f"Vegetation polygons loaded: {len(vegetation)}")
print(f"Suburb CRS: {suburbs.crs}")
print(f"Vegetation CRS: {vegetation.crs}")
suburbs.head(3)

Suburbs loaded: 191
Vegetation polygons loaded: 4074
Suburb CRS: EPSG:7844
Vegetation CRS: EPSG:7844


,SAL_CODE21,SAL_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,SHAPE_Leng,SHAPE_Area,geometry
0,31236,Greenbank,3,Queensland,AUS,Australia,110.5017,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.658446,0.010110,"MULTIPOLYGON (((152.95254 -27.63358, 152.95321..."
1,31067,Forestdale,3,Queensland,AUS,Australia,6.0833,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.122959,0.000556,"MULTIPOLYGON (((152.9841 -27.64863, 152.98425 ..."
2,31639,Larapinta (Qld),3,Queensland,AUS,Australia,5.9297,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.113420,0.000542,"MULTIPOLYGON (((153.01489 -27.64272, 153.01494..."


## 2. Calculate Heat & Canopy Metrics
### 2.1 Mean Land Surface Temperature per Suburb
Using zonal statistics to extract the mean Landsat LST value for each suburb polygon.

In [4]:
# Reproject suburbs to EPSG:4326 to match raster
suburbs_4326 = suburbs.to_crs("EPSG:4326")

with rasterio.open("../data/brisbane_lst.tif") as src:
    lst_data = src.read(1).astype(float)
    lst_data[lst_data > 60] = np.nan
    lst_data[lst_data < 10] = np.nan
    affine = src.transform

stats = zonal_stats(
    suburbs_4326,
    lst_data,
    affine=affine,
    stats=["mean", "max"],
    nodata=np.nan
)

suburbs['lst_mean'] = [s['mean'] if s['mean'] else np.nan for s in stats]
suburbs['lst_max'] = [s['max'] if s['max'] else np.nan for s in stats]

print(f"Temperature range: {suburbs['lst_mean'].min():.1f}°C — {suburbs['lst_mean'].max():.1f}°C")
print(f"Mean across Brisbane: {suburbs['lst_mean'].mean():.1f}°C")
suburbs[['SAL_NAME21','lst_mean','lst_max']].sort_values('lst_mean', ascending=False).head(10)

Temperature range: 27.0°C — 45.2°C
Mean across Brisbane: 35.5°C


,SAL_NAME21,lst_mean,lst_max
172,Geebung,45.241570,55.402009
178,Taigum,45.232116,51.930440
171,Virginia (Qld),44.949706,54.374324
167,Northgate (Qld),44.662449,50.378659
170,Banyo,44.132127,51.261647
162,Hendra,43.778557,56.484382
165,Nundah,43.433921,49.524154
55,Jindalee (Qld),43.119544,43.613258
111,Manly West,42.987956,47.803750
166,Wavell Heights,42.449770,47.198761


### 2.2 Protected Canopy Coverage per Suburb
Calculating the percentage of each suburb's area covered by BCC protected vegetation polygons.
Vegetation polygons are clipped to suburb boundaries before area summation to prevent overlap inflation.

In [5]:
# Reproject to UTM Zone 56S for accurate area in metres
suburbs_projected = suburbs.to_crs("EPSG:32756")
vegetation_projected = vegetation.to_crs("EPSG:32756")

# Clip vegetation to suburb boundaries
vegetation_clipped = gpd.overlay(
    vegetation_projected,
    suburbs_projected[['SAL_NAME21', 'geometry']],
    how='intersection'
)
vegetation_clipped['clipped_area_m2'] = vegetation_clipped.geometry.area

# Sum per suburb
veg_area = vegetation_clipped.groupby('SAL_NAME21')['clipped_area_m2'].sum().reset_index()
veg_area.columns = ['SAL_NAME21', 'veg_area_m2']

# Suburb area and canopy %
suburbs['suburb_area_m2'] = suburbs_projected.geometry.area
suburbs = suburbs.merge(veg_area, on='SAL_NAME21', how='left')
suburbs['veg_area_m2'] = suburbs['veg_area_m2'].fillna(0)
suburbs['canopy_pct'] = (suburbs['veg_area_m2'] / suburbs['suburb_area_m2']) * 100
suburbs['canopy_pct'] = suburbs['canopy_pct'].clip(upper=100)

print(f"Mean canopy coverage: {suburbs['canopy_pct'].mean():.1f}%")
print(f"Max canopy coverage: {suburbs['canopy_pct'].max():.1f}%")
print(f"Min canopy coverage: {suburbs['canopy_pct'].min():.1f}%")
suburbs[['SAL_NAME21','canopy_pct']].sort_values('canopy_pct', ascending=False).head(10)

Mean canopy coverage: 13.7%
Max canopy coverage: 96.7%
Min canopy coverage: 0.0%


,SAL_NAME21,canopy_pct
113,Mount Coot-tha,96.709173
175,Nudgee Beach,95.332294
15,Karawatha,87.178107
7,Drewvale,59.543067
17,Stretton,49.937978
56,Nathan,48.082695
27,Willawong,45.617794
101,Seven Hills (Qld),39.653352
94,Carina Heights,38.048427
48,Burbank,36.275590


## 3. Priority Scoring
Suburbs are scored by combining normalised heat and canopy deficit into a single composite metric.
A high priority score indicates a suburb that is simultaneously hot AND under-canopied — 
the strongest case for new street tree planting.

In [6]:
# Normalise heat 0-1
suburbs['heat_norm'] = (
    (suburbs['lst_mean'] - suburbs['lst_mean'].min()) /
    (suburbs['lst_mean'].max() - suburbs['lst_mean'].min())
)

# Invert canopy — low canopy = high score
suburbs['canopy_norm'] = 1 - (
    (suburbs['canopy_pct'] - suburbs['canopy_pct'].min()) /
    (suburbs['canopy_pct'].max() - suburbs['canopy_pct'].min())
)

# Equal weight priority score
suburbs['priority_score'] = (suburbs['heat_norm'] + suburbs['canopy_norm']) / 2

# Drop suburbs with no temperature data
suburbs_scored = suburbs.dropna(subset=['lst_mean']).copy()
suburbs_scored = suburbs_scored.sort_values('priority_score', ascending=False)

# Display top 20
top20 = suburbs_scored[[
    'SAL_NAME21', 'lst_mean', 'canopy_pct', 'priority_score'
]].head(20).copy()
top20.columns = ['Suburb', 'Mean Temp (°C)', 'Canopy %', 'Priority Score']
top20['Mean Temp (°C)'] = top20['Mean Temp (°C)'].round(1)
top20['Canopy %'] = top20['Canopy %'].round(2)
top20['Priority Score'] = top20['Priority Score'].round(3)
top20.reset_index(drop=True, inplace=True)
top20.index += 1
top20

,Suburb,Mean Temp (°C),Canopy %,Priority Score
1,Taigum,45.2,0.00,1.000
2,Geebung,45.2,4.95,0.974
3,Virginia (Qld),44.9,4.65,0.968
4,Northgate (Qld),44.7,6.71,0.949
5,Hendra,43.8,2.65,0.946
6,Jindalee (Qld),43.1,0.00,0.942
7,Banyo,44.1,8.19,0.927
8,Ormiston,41.9,0.00,0.909
9,Archerfield,42.1,0.75,0.909
10,Wellington Point,41.6,0.00,0.901


## 3.1 Population-Weighted Sensitivity Analysis
Raw heat + canopy scoring treats all suburbs equally regardless of how many people live there.
A budget-conscious planner should prioritise suburbs where tree planting benefits the most residents.
We introduce population density as a third variable to test how rankings shift.

In [9]:
# Debug — check what codes look like on both sides
print("Census codes sample:", pop['SAL_CODE21'].head(5).tolist())
print("Census code type:", pop['SAL_CODE21'].dtype)
print()
print("Suburb codes sample:", suburbs_scored['SAL_CODE21'].head(5).tolist())
print("Suburb code type:", suburbs_scored['SAL_CODE21'].dtype)
print()

# Check if any match at all
matches = pop['SAL_CODE21'].isin(suburbs_scored['SAL_CODE21'])
print(f"Matching codes: {matches.sum()} out of {len(pop)}")

Census codes sample: ['30001', '30002', '30003', '30004', '30005']
Census code type: str

Suburb codes sample: ['32717', '31108', '32949', '32188', '31320']
Suburb code type: str

Matching codes: 191 out of 3235


In [10]:
# Reset to avoid duplicate column issues from previous runs
suburbs_scored = suburbs.dropna(subset=['lst_mean']).copy()
suburbs_scored = suburbs_scored.sort_values('priority_score', ascending=False)

# Load census population data
pop = pd.read_csv("../data/2021Census_G01_QLD_SAL.csv", 
                   usecols=['SAL_CODE_2021', 'Tot_P_P'])

# Strip SAL prefix
pop['SAL_CODE21'] = pop['SAL_CODE_2021'].str.replace('SAL', '').str.strip()
pop = pop[['SAL_CODE21', 'Tot_P_P']]
pop.columns = ['SAL_CODE21', 'population']

# Ensure same type
suburbs_scored['SAL_CODE21'] = suburbs_scored['SAL_CODE21'].astype(str).str.strip()

# Merge
suburbs_scored = suburbs_scored.merge(pop, on='SAL_CODE21', how='left')

# Calculate population density
suburbs_scored['area_km2'] = suburbs_scored['suburb_area_m2'] / 1_000_000
suburbs_scored['pop_density'] = suburbs_scored['population'] / suburbs_scored['area_km2']

print(f"Population merged for {suburbs_scored['population'].notna().sum()} suburbs")
print(f"Mean population: {suburbs_scored['population'].mean():.0f}")
print()
print("Most populous suburbs:")
suburbs_scored[['SAL_NAME21','population','pop_density']].sort_values(
    'population', ascending=False).head(10)

Population merged for 191 suburbs
Mean population: 7402

Most populous suburbs:


,SAL_NAME21,population,pop_density
66,Forest Lake,22676,2624.737657
19,Thornlands,19263,3001.214719
120,Coorparoo,18132,3406.178211
53,Sunnybank Hills,18085,2795.236210
21,Capalaba,18002,963.258112
43,Calamvale,17994,2707.858656
174,The Gap (Brisbane - Qld),17318,6028.559556
84,Carindale,16535,1549.010281
18,Alexandra Hills,16472,1205.511526
61,Albany Creek,16385,2522.895213


### Scenario B — Population-Weighted Priority Score
Introducing population density as a third variable rewards suburbs where tree planting 
benefits the greatest number of residents per km². A suburb that is hot and treeless 
but largely uninhabited ranks lower than a dense residential suburb with the same conditions.

In [11]:
# Normalise population density 0-1
suburbs_scored['popden_norm'] = (
    (suburbs_scored['pop_density'] - suburbs_scored['pop_density'].min()) /
    (suburbs_scored['pop_density'].max() - suburbs_scored['pop_density'].min())
)

# Scenario B — equal weight across three variables
suburbs_scored['priority_score_weighted'] = (
    suburbs_scored['heat_norm'] + 
    suburbs_scored['canopy_norm'] + 
    suburbs_scored['popden_norm']
) / 3

suburbs_scored = suburbs_scored.sort_values('priority_score_weighted', ascending=False)

# Build comparison table — Scenario A vs Scenario B
scenario_a = suburbs_scored.sort_values('priority_score', ascending=False)[
    ['SAL_NAME21','priority_score']].head(20).reset_index(drop=True)
scenario_a.index += 1
scenario_a.columns = ['Suburb (Scenario A)', 'Score A']

scenario_b = suburbs_scored.sort_values('priority_score_weighted', ascending=False)[
    ['SAL_NAME21','priority_score_weighted']].head(20).reset_index(drop=True)
scenario_b.index += 1
scenario_b.columns = ['Suburb (Scenario B)', 'Score B']

comparison = pd.concat([scenario_a, scenario_b], axis=1)
comparison['Score A'] = comparison['Score A'].round(3)
comparison['Score B'] = comparison['Score B'].round(3)

print("=== SCENARIO COMPARISON ===")
print("Scenario A: Heat + Canopy Deficit only")
print("Scenario B: Heat + Canopy Deficit + Population Density")
print()
comparison

=== SCENARIO COMPARISON ===
Scenario A: Heat + Canopy Deficit only
Scenario B: Heat + Canopy Deficit + Population Density



,Suburb (Scenario A),Score A,Suburb (Scenario B),Score B
1,Taigum,1.000,Jindalee (Qld),0.961
2,Geebung,0.974,Taigum,0.674
3,Virginia (Qld),0.968,Geebung,0.650
4,Northgate (Qld),0.949,Virginia (Qld),0.645
5,Hendra,0.946,Northgate (Qld),0.633
6,Jindalee (Qld),0.942,Hendra,0.631
7,Banyo,0.927,Banyo,0.618
8,Ormiston,0.909,Ormiston,0.608
9,Archerfield,0.909,Archerfield,0.606
10,Wellington Point,0.901,Wellington Point,0.601


### Methodology Note — Daytime Population Exposure

Residential population (Scenario B) is a conservative measure of heat exposure. 
Suburbs like Fortitude Valley and Eagle Farm have significant daytime worker populations 
that residential counts do not capture.

**Attempted:** ABS Census G62 (Method of Travel to Work) as a proxy for inbound workers. 
**Finding:** G62 is origin-based — it counts residents leaving each suburb for work, 
not workers arriving. It cannot be used to estimate inbound daytime population.


**Conclusion:** Inbound worker counts at suburb (SAL) level are not available in the 
2021 Census DataPack. A complete daytime exposure analysis would require:
- ABS Place of Work data at SA2 level (coarser geography)
- Or council-level employment land use data from Brisbane City Plan

This represents an open opportunity for further analysis. The current scoring 
(Scenario B) uses residential population density as the best available proxy.

## 4. Interactive Map — Priority Suburbs for Street Tree Planting
Choropleth map showing priority score across Brisbane suburbs.
Darker red = higher priority (hot + low canopy). Click any suburb for details.

In [17]:
import folium
from folium.features import GeoJsonTooltip

# Reproject to WGS84 for Folium (requires EPSG:4326)
suburbs_map = suburbs_scored.to_crs("EPSG:4326")

# Initialise map centred on Brisbane
m = folium.Map(
    location=[-27.47, 153.02],
    zoom_start=11,
    tiles='CartoDB positron'
)

# Choropleth layer — priority score
folium.Choropleth(
    geo_data=suburbs_map.__geo_interface__,
    data=suburbs_scored,
    columns=['SAL_NAME21', 'priority_score'],
    key_on='feature.properties.SAL_NAME21',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Priority Score (Heat + Canopy Deficit)',
    nan_fill_color='lightgrey'
).add_to(m)

# Tooltip layer — clickable suburb details
folium.GeoJson(
    suburbs_map.__geo_interface__,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'transparent'
    },
    tooltip_kwds=dict(style="background-color: #16213e; color: #ffffff; font-family: 'Segoe UI'; font-size: 13px; border: 1px solid #e74c3c;"),
    tooltip=GeoJsonTooltip(
        fields=['SAL_NAME21', 'lst_mean', 'canopy_pct', 
                'population', 'priority_score'],
        aliases=['Suburb', 'Mean Temp (°C)', 'Canopy %', 
                 'Population', 'Priority Score'],
        localize=True,
        sticky=True,
        style="""
            background-color: #16213e;
            color: #ffffff;
            font-family: 'Segoe UI', Arial, sans-serif;
            font-size: 13px;
            border: 1px solid #e74c3c;
            border-radius: 4px;
            padding: 8px;
        """
    )
).add_to(m)

import json

# Get top 20 for the panel
top20_panel = suburbs_scored.sort_values('priority_score', ascending=False)[
    ['SAL_NAME21', 'lst_mean', 'canopy_pct', 'priority_score']
].head(20).copy()
top20_panel['lst_mean'] = top20_panel['lst_mean'].round(1)
top20_panel['canopy_pct'] = top20_panel['canopy_pct'].round(1)
top20_panel['priority_score'] = top20_panel['priority_score'].round(3)

# Build table rows HTML
table_rows = ""
for i, (_, row) in enumerate(top20_panel.iterrows(), 1):
    table_rows += f"""
    <tr>
        <td style="padding:4px 8px;font-weight:bold;color:#e74c3c">{i}</td>
        <td style="padding:4px 8px">{row['SAL_NAME21']}</td>
        <td style="padding:4px 8px;text-align:center">{row['lst_mean']}°C</td>
        <td style="padding:4px 8px;text-align:center">{row['canopy_pct']}%</td>
        <td style="padding:4px 8px;text-align:center;font-weight:bold">{row['priority_score']}</td>
    </tr>"""

# Get folium map HTML
map_html = m.get_root().render()

# Build combined layout
combined_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Brisbane Urban Heat & Canopy Gap Analysis</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #1a1a2e; color: #eee; }}
        
        .header {{
            background: linear-gradient(135deg, #1a1a2e, #16213e);
            padding: 20px 30px;
            border-bottom: 2px solid #e74c3c;
        }}
        .header h1 {{ 
            font-size: 1.4em; 
            color: #fff;
            letter-spacing: 1px;
        }}
        .header p {{ 
            font-size: 0.8em; 
            color: #aaa; 
            margin-top: 4px; 
        }}
        
        .container {{
            display: flex;
            height: calc(100vh - 80px);
        }}
        
        .side-panel {{
            width: 340px;
            background: #16213e;
            overflow-y: auto;
            border-right: 1px solid #2d2d5e;
            flex-shrink: 0;
        }}
        
        .map-panel {{
            flex: 1;
            position: relative;
        }}
        
        .map-panel iframe {{
            width: 100%;
            height: 100%;
            border: none;
        }}
        
        .panel-title {{
            padding: 14px 16px;
            font-size: 0.85em;
            font-weight: bold;
            text-transform: uppercase;
            letter-spacing: 1px;
            color: #e74c3c;
            border-bottom: 1px solid #2d2d5e;
            background: #0f3460;
        }}
        
        table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 0.78em;
        }}
        
        thead th {{
            padding: 8px;
            background: #0f3460;
            color: #aaa;
            font-weight: normal;
            text-transform: uppercase;
            font-size: 0.75em;
            letter-spacing: 0.5px;
            position: sticky;
            top: 0;
        }}
        
        tbody tr:nth-child(even) {{
            background: rgba(255,255,255,0.03);
        }}
        
        tbody tr:hover {{
            background: rgba(231,76,60,0.15);
        }}
        
        td {{ color: #ddd; border-bottom: 1px solid #2d2d5e22; }}
        
        .methodology {{
            padding: 12px 16px;
            font-size: 0.72em;
            color: #888;
            border-top: 1px solid #2d2d5e;
            line-height: 1.5;
        }}
        
        .methodology strong {{ color: #aaa; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>🌳 Brisbane Urban Heat & Canopy Gap Analysis</h1>
        <p>Priority suburbs for street tree planting · Landsat 8 LST (summers 2022–2024) · BCC Protected Vegetation · ABS Census 2021</p>
    </div>
    
    <div class="container">
        <div class="side-panel">
            <div class="panel-title">🔴 Top 20 Priority Suburbs</div>
            <table>
                <thead>
                    <tr>
                        <th>#</th>
                        <th>Suburb</th>
                        <th>Temp</th>
                        <th>Canopy</th>
                        <th>Score</th>
                    </tr>
                </thead>
                <tbody>
                    {table_rows}
                </tbody>
            </table>
            <div class="methodology">
                <strong>Methodology</strong><br>
                Priority Score = (Normalised Heat + Canopy Deficit) / 2<br><br>
                <strong>Data Sources</strong><br>
                · LST: Landsat 8 C2 via Google Earth Engine<br>
                · Canopy: BCC NALL 2003 Protected Vegetation<br>
                · Boundaries: ABS SAL 2021 GDA2020<br><br>
                <strong>Limitation</strong><br>
                Canopy coverage reflects protected vegetation only — 
                total urban canopy is likely higher.
            </div>
        </div>
        
        <div class="map-panel">
            {map_html}
        </div>
    </div>
</body>
</html>
"""

# Save
with open("../outputs/brisbane_heat_priority_dashboard.html", "w", encoding="utf-8") as f:
    f.write(combined_html)

print("Dashboard saved to outputs/brisbane_heat_priority_dashboard.html")

Dashboard saved to outputs/brisbane_heat_priority_dashboard.html


## 5. Conclusions

This analysis combined three open datasets: Landsat 8 satellite imagery, BCC protected 
vegetation polygons, and ABS Census boundaries, to produce a reproducible, data-driven 
prioritisation of Brisbane suburbs for street tree investment.

### Key Findings

**The northern industrial corridor demands attention.**
Taigum, Geebung, Virginia, Northgate, and Banyo consistently rank as the highest priority 
suburbs across all scoring scenarios. These suburbs share high summer land surface 
temperatures (43–45°C mean), minimal protected canopy coverage (0–9%), and moderate 
residential populations — a combination that represents a clear and actionable gap.

**Jindalee is the standout finding when citizen welfare is prioritised.**
Under the population-weighted scenario (Scenario B), Jindalee rises to #1. It is a dense 
residential suburb with high heat and zero recorded protected canopy — meaning its 
residents bear disproportionate heat exposure with no vegetative relief. This finding 
would not surface from visual inspection of a heat map alone.

**Industrial suburbs (Eagle Farm, Archerfield, Brisbane Airport) score high on heat 
and canopy deficit but have negligible residential populations.** These should be 
deprioritised for street tree planting in favour of residential suburbs with equivalent 
or near-equivalent scores.

**The top 20 is consistent across methodologies.** The same suburbs appear regardless of 
whether population weighting is applied, suggesting the heat and canopy signal is 
strong enough to be methodology-independent.

### Limitations

| Limitation | Impact | Potential Resolution |
|---|---|---|
| Canopy from NALL 2003 protected vegetation only | Undercounts total urban canopy | Use LiDAR-derived canopy height model |
| Suburb boundary approximation (bounding box) | Minor fringe suburb inclusion | Use LGA boundary file |
| LST from summers 2022–2024 only | May not capture long-term trends | Extend to 10-year composite |
| Residential population only | Underestimates daytime exposure in commercial suburbs | ABS Place of Work data at SA2 level |
| Inbound worker data unavailable at SAL level | Daytime exposure unquantified | Future work — see open contribution note |

### Open Contribution

The daytime population gap is the most significant unresolved limitation. 
Inbound worker counts at suburb level are not published in the ABS 2021 Census DataPack. 
A complete daytime exposure analysis would require either ABS Place of Work data 
at SA2 level or council employment land use data from Brisbane City Plan 2014.

**This repository is open for contribution.** If you have access to or can derive 
inbound worker estimates at suburb level, the scoring pipeline is designed to 
accept a fourth normalised variable with minimal modification.

## 6. ArcGIS Online Integration
Publishing the priority scoring results as a hosted feature layer on ArcGIS Online
for interactive visualisation in Map Viewer and Scene Viewer.

In [18]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
import os

# Connect to ArcGIS Online — will prompt for password
gis = GIS("https://www.arcgis.com", "Jlopez94")
print(f"Connected as: {gis.properties.user.username}")

Enter password:  ········


Connected as: Jlopez94


In [19]:
import tempfile
import os

# Save scored suburbs as GeoJSON for upload
geojson_path = "../outputs/brisbane_scored_upload.geojson"

# Select only the columns we need — keeps the file clean
cols_to_publish = [
    'SAL_CODE21', 'SAL_NAME21', 'lst_mean', 'lst_max',
    'canopy_pct', 'population', 'pop_density',
    'heat_norm', 'canopy_norm', 'priority_score',
    'priority_score_weighted', 'area_km2', 'geometry'
]

suburbs_publish = suburbs_scored[cols_to_publish].to_crs("EPSG:4326")
suburbs_publish['lst_mean'] = suburbs_publish['lst_mean'].round(2)
suburbs_publish['lst_max'] = suburbs_publish['lst_max'].round(2)
suburbs_publish['canopy_pct'] = suburbs_publish['canopy_pct'].round(2)
suburbs_publish['priority_score'] = suburbs_publish['priority_score'].round(4)
suburbs_publish['priority_score_weighted'] = suburbs_publish['priority_score_weighted'].round(4)
suburbs_publish['pop_density'] = suburbs_publish['pop_density'].round(1)

suburbs_publish.to_file(geojson_path, driver='GeoJSON')
print(f"GeoJSON saved: {geojson_path}")
print(f"Features: {len(suburbs_publish)}")
print(f"Columns: {[c for c in cols_to_publish if c != 'geometry']}")

GeoJSON saved: ../outputs/brisbane_scored_upload.geojson
Features: 191
Columns: ['SAL_CODE21', 'SAL_NAME21', 'lst_mean', 'lst_max', 'canopy_pct', 'population', 'pop_density', 'heat_norm', 'canopy_norm', 'priority_score', 'priority_score_weighted', 'area_km2']


In [20]:
# Upload GeoJSON as item
item_properties = {
    'title': 'Brisbane Urban Heat & Canopy Priority Scoring',
    'description': (
        'Priority scoring for street tree planting across Brisbane suburbs. '
        'Derived from Landsat 8 Land Surface Temperature (GEE, summers 2022-2024), '
        'BCC Protected Vegetation (NALL 2003), and ABS Census 2021. '
        'Priority Score = (Normalised Heat + Canopy Deficit) / 2.'
    ),
    'tags': 'Brisbane, urban heat, canopy, street trees, GIS, Landsat, ABS, priority',
    'type': 'GeoJson',
    'licenseInfo': 'Open data — sources: USGS, BCC, ABS'
}

# Upload the GeoJSON
geojson_item = gis.content.add(
    item_properties=item_properties,
    data=geojson_path
)
print(f"Uploaded: {geojson_item.title}")
print(f"Item ID: {geojson_item.id}")

# Publish as hosted feature layer
feature_layer_item = geojson_item.publish()
print(f"Published as feature layer: {feature_layer_item.title}")
print(f"Layer URL: {feature_layer_item.url}")
print(f"ArcGIS Online item: https://www.arcgis.com/home/item.html?id={feature_layer_item.id}")

C:\Users\jlope\Documents\Projects\brisbane-heat-analysis\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\jlope\Documents\Projects\brisbane-heat-analysis\venv\Lib\site-packages\arcgis\graph\data_model_types.py:82: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class GraphProperty(BaseModel):
C:\Users\jlope\Documents\Projects\brisbane-heat-analysis\venv\Lib\site-packages\arcgis\graph\data_model_types.py:285: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migra

Uploaded: Brisbane Urban Heat & Canopy Priority Scoring
Item ID: eb68cc9fac494ceb9639ca871ffac3fc


Exception: Unable to publish item.
User 'Jlopez94' does not have publishing privileges.
(Error Code: 400)